In [0]:
silver_path = "abfss://silver-layer@stockdatalake1.dfs.core.windows.net/stock"

df = spark.read.format("delta").load(silver_path)

In [0]:
df_daily = (
    df.groupBy("date")
    .agg(
        F.avg("Close").alias("avg_close"),
        F.max("High").alias("max_high"),
        F.min("Low").alias("min_low"),
        F.sum("shares_traded").alias("total_volume"),
        F.sum("Turnover_cr").alias("total_turnover")
    )
)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gold_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold_catalog.gold_schema;

In [0]:
df_daily.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_catalog.gold_schema.gold_daily")

In [0]:
df_weekly = (
    df
    .withColumn("week", F.weekofyear("date"))
    .withColumn("year", F.year("date"))
    .groupBy("year", "week")
    .agg(
        F.avg("Close").alias("avg_close"),
        F.max("High").alias("max_high"),
        F.min("Low").alias("min_low"),
        F.sum("shares_traded").alias("total_volume"),
        F.sum("Turnover_cr").alias("total_turnover")
    )
)

In [0]:
df_weekly.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_catalog.gold_schema.gold_weekly")

In [0]:
df_monthly = (
    df.groupBy("year", "month")
    .agg(
        F.avg("Close").alias("avg_close"),
        F.max("High").alias("max_high"),
        F.min("Low").alias("min_low"),
        F.sum("shares_traded").alias("total_volume"),
        F.sum("Turnover_cr").alias("total_turnover")
    )
)

In [0]:
df_monthly.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_catalog.gold_schema.gold_monthly")

In [0]:
gold_base_path = "abfss://gold-layer@stockdatalake1.dfs.core.windows.net/"

df_monthly.write.mode("overwrite").format("delta").save(gold_base_path + "monthly")
df_weekly.write.mode("overwrite").format("delta").save(gold_base_path + "weekly")
df_daily.write.mode("overwrite").format("delta").save(gold_base_path + "daily")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


df_monthly_perf = df_monthly.withColumn(
    "pct_change",
    ((F.col("max_high") - F.col("min_low")) / F.col("min_low")) * 100
)


top_gainers_monthly = df_monthly_perf.orderBy(F.col("pct_change").desc()).limit(10)


top_gainers_monthly.show()

In [0]:
top_gainers_monthly.write \
    .mode("overwrite") \
    .format("delta") \
    .option("path", gold_base_path + "top_gainers") \
    .saveAsTable("gold_catalog.gold_schema.top_gainers")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


df_monthly_perf = df_monthly.withColumn(
    "pct_change",
    ((F.col("max_high") - F.col("min_low")) / F.col("min_low")) * 100
)


top_losers_monthly = df_monthly_perf.orderBy(F.col("pct_change").asc()).limit(10)


top_losers_monthly.show()

In [0]:
top_losers_monthly.write \
    .mode("overwrite") \
    .format("delta") \
    .option("path", gold_base_path + "top_losers") \
    .saveAsTable("gold_catalog.gold_schema.top_losers")

In [0]:
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_catalog.gold_schema.df_gold")

In [0]:
df_gold.display()

In [0]:

df.write.mode("overwrite").saveAsTable("gold_catalog.gold_schema.gold_table")

In [0]:

spark.sql("SHOW TABLES").show()

spark.sql("SELECT * FROM gold_table LIMIT 10").show()